# Optional Deep-Dive: Transformers From Scratch

> **This is an optional supplementary notebook.** The main course path does not
> require it. The transformer concept is taught as a short mini-lesson in the
> required topics. This notebook is the from-scratch BUILD, for learners who want
> to see the internals. You can complete every required topic without opening it.

### Who this is for

This deep-dive is for developers who want to see what is actually inside a
Transformer. You will build the architecture behind BERT, GPT, and T5 module by
module in PyTorch, then train it on a remote GPU. If you are happy treating models
as black boxes you can safely skip this. If you are the kind of engineer who wants
to open the box, read on.

### Is it self-contained?

Yes. This notebook does not depend on any other notebook in the course. Comfort
with the idea of attention helps, but a full recap of scaled dot-product attention
and of `torch.nn.MultiheadAttention` is included below, so you can follow even if
you have never seen attention before.

### What you need

A SageMaker notebook kernel for the architecture demos (CPU is fine). The capstone
at the end submits one short remote GPU training job (ml.g4dn.xlarge); that step
needs working AWS / SageMaker credentials and is flagged with its own prerequisite
note and a guard cell. The capstone is optional within this optional notebook: the
architecture is fully built and tested before any remote job is submitted.

## Transformers and a Translator, Built From Scratch

A self-contained build of the Transformer architecture, framed around a Barclays
Customer Support translation scenario.

## What you will build
This notebook restates everything it needs as it goes, including a recap of
scaled dot-product attention and of PyTorch's multi-head attention module. In it
you will:
1. Understand WHY a sequence model that reads tokens one at a time struggles on
   long sequences
2. Build the full Transformer architecture from scratch in PyTorch
3. Combine your components into a Translator model
4. Submit a remote GPU training job on SageMaker

## Learning objectives
1. Articulate the bottlenecks that the Transformer architecture solves
2. Implement positional encoding and multi-head self-attention in PyTorch
3. Assemble encoder, decoder, and full Transformer from modular components
4. Submit a SageMaker PyTorch training job and retrieve model artifacts

## Why would a model USER want to understand the internals?

Most of the time you will use Transformers, not build them. You load a pre-trained
model and call it. So why spend an afternoon building one by hand?

Because the internals explain the behaviour you will debug in production:

- **Context windows and cost.** Attention compares every token to every other
  token. Once you have built that comparison yourself, the quadratic cost of long
  prompts stops being a mystery and becomes an obvious consequence of the design.
- **Why position matters.** Models can be sensitive to where information sits in a
  prompt. That is not a bug; it is positional encoding doing its job. You will see
  exactly why by removing it and watching the model go order-blind.
- **Encoder vs decoder vs encoder-decoder.** BERT-style, GPT-style, and T5-style
  models differ in which half of this architecture they keep. Knowing the halves
  tells you which model class fits which task.
- **Fine-tuning intuition.** When you later freeze layers, attach adapters, or
  pick a learning rate, you are reasoning about the exact blocks you are about to
  build.

You do not need this to use a model. You need it to use a model well, and to know
what to try when one misbehaves. That is the whole reason this optional notebook
exists.

## Section 0 - Environment Setup

## How this notebook is organised

This is a standalone deep-dive, so it does not sit at a fixed point in any course
sequence. It runs front to back on its own:

- Section 1 - why a token-at-a-time sequence model struggles, plus a recap of
  scaled dot-product attention and PyTorch multi-head attention
- Section 2 - positional encoding, plus Lab 1
- Section 3 - building the Transformer block by block, plus Lab 2
- Section 4 - the capstone: training the Translator on a remote GPU

Run the cells in order. Each section restates what it needs from the section
before it.

In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Optional deep-dive: Transformers from scratch.
# Architecture demos run in this kernel (CPU is fine); the capstone submits a
# remote GPU job on ml.g4dn.xlarge.

!pip install -q \
    "sagemaker>=2.200.0,<3.0.0" \
    "numpy<2" \
    "matplotlib>=3.7.0"

print("RESTART KERNEL before continuing -- environment packages were installed/upgraded.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math
import os
import random
import warnings

import sagemaker
from sagemaker import get_execution_role
import boto3

warnings.filterwarnings("ignore")

# Reproducibility - set_seeds is defined here for reproducibility
def set_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# Device check - training cells run on CPU here; capstone trains on remote GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device (notebook kernel): {device}")
print()

# SageMaker session - needed for the capstone remote training job
sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"SageMaker role: {role}")
print(f"Default bucket: {bucket}")
print(f"Region: {region}")

## What are we building?

Earlier sequence models processed text with a recurrent network (an RNN): one
token at a time, each step feeding the next. Attention was bolted on top to let
the model focus on relevant earlier tokens, but the recurrent network was still
the bottleneck.

The Transformer throws the recurrent network away entirely and uses attention for
everything. The result is the architecture behind BERT, GPT, T5, and every modern
large language model you have heard of.

By the end of this notebook you will have:
1. A working Transformer built module by module in PyTorch
2. A translator that maps Spanish complaint tickets to English
3. A remote GPU training job running on SageMaker ml.g4dn.xlarge

The Barclays scenario: the complaints team receives tickets in Spanish from
Latin American customers. We will build a translator to route those tickets
to the English-speaking review team.

## Recap: scaled dot-product attention, and PyTorch's multi-head attention

Everything in this notebook is built on one operation. If you have seen it before,
skim this. If you have not, this is all you need.

Attention answers: for a given token, which other tokens should it look at, and
how much? It works with three vectors per token:

- **Query (Q)** - what this token is looking for
- **Key (K)** - what each token offers, used for matching
- **Value (V)** - the actual information each token carries

The computation, for sequences packed into matrices:

```
Attention(Q, K, V) = softmax( (Q K^T) / sqrt(d_k) ) V
```

Step by step:

1. `Q K^T` scores every query against every key. For a sequence of length T this
   is a T-by-T matrix: row i, column j is "how much should token i attend to
   token j".
2. Divide by `sqrt(d_k)` (the key dimension). Without this the scores grow large,
   softmax saturates, and gradients shrink. This is the "scaled" part.
3. `softmax` over each row turns the scores into weights that sum to 1.
4. Multiply by `V`: each token's output is a weighted blend of every token's
   value.

**Self-attention** is the case where Q, K, and V all come from the same sequence:
every token attends to every token in its own sequence. **Cross-attention** is
when Q comes from one sequence and K, V come from another; that is how a decoder
reads the encoder's output. **Multi-head attention** runs several of these in
parallel on different learned projections and concatenates the results, so the
model can attend in several ways at once.

### PyTorch gives you this as a module: torch.nn.MultiheadAttention

You do not have to hand-roll the four steps above every time. PyTorch ships the
whole multi-head operation as `torch.nn.MultiheadAttention`. This notebook uses
it as the attention building block, so here is the contract before you see it in
action:

- Construct it with `nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)`.
  `embed_dim` is your model width (`d_model`); `num_heads` must divide `embed_dim`.
  `batch_first=True` means tensors are shaped `(batch, seq_len, d_model)`.
- Call it as `output, weights = mha(query, key, value)`. For self-attention you
  pass the same tensor three times: `mha(x, x, x)`.
- `output` has the same shape as `query`. `weights` is the
  `(batch, seq_len, seq_len)` attention matrix - the softmax table from step 3.
- Optional arguments matter later: `key_padding_mask` ignores padding tokens, and
  `attn_mask` (a causal mask) stops a decoder token from peeking at future tokens.

The very next demo builds an `nn.MultiheadAttention` and runs self-attention on a
short sequence so you can see the attention matrix directly. Every encoder and
decoder layer you build afterwards is just this module plus a feed-forward network
and some layer norms. That is the whole foundation.

## Beat 1 -- Section 1 - Why Transformers? The problem with a recurrent sequence model

As the recap noted, attention helps a decoder focus on relevant earlier states.
But if you keep a recurrent network around the outside, two problems remain:

1. Sequential processing: the recurrent network processes tokens one at a time.
   Token 50 cannot be computed until token 49 is done.
   No parallelism, so training is slow on long sequences.

2. Long-range dependencies: even with attention, the hidden state that "carries"
   information from token 1 to token 50 can get diluted through 49 matrix
   multiplications.

Let us see the long-range problem concretely before we fix it.

In [ ]:
# Beat 1: Hidden state dilution in a simple RNN.
# We show that the gradient of the final hidden state w.r.t. an early token
# decays exponentially with distance - even with tanh activations.
#
# This is the "vanishing gradient" problem that motivated Transformers.

set_seeds(42)

# Simulate a one-layer vanilla RNN over a complaint sequence.
# W_h: hidden-to-hidden weight matrix
# The recurrence is h_t = tanh(W_h @ h_{t-1} + W_x @ x_t)

seq_len = 30
d_h = 64

W_h = torch.randn(d_h, d_h) * 0.5  # small init to avoid explosion
W_x = torch.randn(d_h, d_h) * 0.1

# Gradient of h_T w.r.t. each h_t decays as ||W_h||^(T-t).
# We track the spectral norm as a proxy.
spectral_radius = torch.linalg.norm(W_h).item()

print("Approximate gradient norm ||dh_T / dh_t|| for a vanilla RNN")
print(f"(seq_len={seq_len}, hidden_dim={d_h}, ||W_h||={spectral_radius:.3f})")
print()
print(f"{'Token t':>8}  {'Grad norm':>14}  {'Status':>20}")
print("-" * 50)

for t in [0, 5, 10, 15, 20, 25, 29]:
    g = spectral_radius ** (seq_len - 1 - t)
    status = "STRONG" if g > 0.1 else ("weak" if g > 0.001 else "VANISHED")
    print(f"{t:>8}  {g:>14.6e}  {status:>20}")

print()
print("Token 0 (first word) barely reaches the final hidden state.")
print("Attention alone on the hidden state does not fix the underlying gradient decay.")
print("FIX: process all positions in PARALLEL with attention - no recurrence at all.")

<!-- DIAGRAM: Full encoder-decoder transformer architecture. Left branch: encoder stack with positional encoding, N stacked encoder layers (each with multi-head self-attention, add+norm, feed-forward, add+norm). Right branch: decoder stack with positional encoding, N stacked decoder layers (each with masked multi-head self-attention, add+norm, cross-attention attending to encoder output, add+norm, feed-forward, add+norm). Final linear + softmax at the top. Data flow arrows showing how the encoder output connects to every decoder layer via cross-attention. -->

```mermaid
graph TD
    subgraph encoder ["Encoder Stack (repeated N times)"]
        src["Source tokens\n(complaint text)"] --> pe_enc["Positional Encoding"]
        pe_enc --> mhsa["Multi-Head Self-Attention"]
        mhsa --> an1["Add + LayerNorm"]
        an1 --> ffn_enc["Feed-Forward Network"]
        ffn_enc --> an2["Add + LayerNorm"]
        an2 --> enc_out["Encoder Output\n[batch, T_src, d_model]"]
    end

    subgraph decoder ["Decoder Stack (repeated N times)"]
        tgt["Target tokens\n(translation so far)"] --> pe_dec["Positional Encoding"]
        pe_dec --> mmhsa["Masked Multi-Head\nSelf-Attention"]
        mmhsa --> an3["Add + LayerNorm"]
        an3 --> cross["Cross-Attention\n(queries from decoder,\nkeys+values from encoder)"]
        enc_out -->|"keys + values"| cross
        cross --> an4["Add + LayerNorm"]
        an4 --> ffn_dec["Feed-Forward Network"]
        ffn_dec --> an5["Add + LayerNorm"]
    end

    an5 --> linear["Linear projection\n[d_model -> vocab_size]"]
    linear --> softmax["Softmax\n-> next token probabilities"]

    style enc_out fill:#ddf,stroke:#33c
    style cross fill:#fda,stroke:#c63
    style softmax fill:#dfd,stroke:#3a3
```

*Figure: The diagram shows the full Transformer from the "Attention Is All You Need" paper (Vaswani et al. 2017). Every component we are about to build appears here - encoder on the left, decoder on the right, cross-attention connecting them in the middle.*

In [ ]:
# Beat 3: Self-attention processes all positions in parallel - no hidden state bottleneck.
# We demonstrate that every token can directly attend to every other token in one step.
# Compare this to the RNN above where token 0 needed 29 matrix multiplications to reach
# the final hidden state.

set_seeds(42)

# A minimal self-attention layer (no positional encoding yet - that comes in Section 2).
# We use PyTorch's built-in MultiheadAttention with a single head for clarity.
d_model = 32
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, batch_first=True)
mha.eval()

# Simulate a short complaint: "my account was charged incorrectly last month"
# 8 tokens, d_model=32
seq_len = 8
with torch.no_grad():
    x = torch.randn(1, seq_len, d_model)           # (batch=1, seq_len=8, d_model=32)
    attn_out, attn_weights = mha(x, x, x)          # Q=K=V=x (self-attention)

print("Beat 3: Multi-head self-attention - all positions attend to all positions at once.")
print(f"Input shape:          {x.shape}")
print(f"Output shape:         {attn_out.shape}  (same shape - parallel transform)")
print(f"Attention weights:    {attn_weights.shape}  (seq_len x seq_len matrix)")
print()
print("Each position in the output is a weighted sum of ALL input positions.")
print("No sequential pass required. Token 0 directly sees token 7 with zero intermediate steps.")
print()
print("Attention weight matrix (token i attends to token j):")
weights = attn_weights[0].detach().numpy()
for i in range(seq_len):
    row = "  ".join(f"{w:.2f}" for w in weights[i])
    print(f"  token {i}: [{row}]")
print()
print("Every row sums to 1.0 (softmax). Every token looks at every other token - in parallel.")

## Beat 4 - Observe the Attention Pattern (3 min)

Look at the attention weight matrix printed above and discuss with a partner:

1. Which token received the highest total attention (column sums)? Does that surprise you
   given the weights are random at this point?

2. In an RNN the gradient from token 7 reaching token 0 decayed exponentially.
   In self-attention, what is the path length from token 7 to token 0?

3. The attention matrix is 8 x 8 here. If the sequence were 1,000 tokens long,
   how large would the matrix be? What does that mean for memory as sequences get longer?

## Section 2 - Positional Encoding

Attention is order-agnostic: if you shuffle the input tokens, the attention scores
change but there is no positional bias built in. We need to inject position information
before the first attention layer.

The original Transformer used sinusoidal positional encodings - a deterministic
function that maps each (position, dimension) pair to a unique value. Let us build it.

In [ ]:
# Beat 1: Without positional encoding the Transformer is order-blind.
# The same set of tokens in different order produces the same set of outputs --
# only the arrangement changes. We demonstrate this concretely.

set_seeds(42)

d_model = 32
nhead   = 4

layer_no_pos = nn.TransformerEncoderLayer(
    d_model=d_model, nhead=nhead, dim_feedforward=128, batch_first=True
)
layer_no_pos.eval()

emb = nn.Embedding(10, d_model)
emb.eval()

# Two sequences: same tokens, reversed order
seq_a = torch.tensor([[1, 2, 3, 4, 5]])   # "charge account fraud refund dispute"
seq_b = torch.tensor([[5, 4, 3, 2, 1]])   # reversed

with torch.no_grad():
    out_a = layer_no_pos(emb(seq_a))   # (1, 5, d_model)
    out_b = layer_no_pos(emb(seq_b))

norm_a = out_a[0].norm(dim=-1).detach().numpy()
norm_b = out_b[0].norm(dim=-1).detach().numpy()

print("Beat 1: A Transformer without positional encoding cannot distinguish token order.")
print("=" * 64)
print(f"Sequence A: {seq_a.tolist()}")
print(f"Sequence B: {seq_b.tolist()} (reversed)")
print()
print("Output L2 norms per position:")
print(f"  Seq A: {norm_a.round(3)}")
print(f"  Seq B: {norm_b.round(3)}")
print()
sorted_a = sorted(norm_a.tolist())
sorted_b = sorted(norm_b.tolist())
print("Sorted norms -- same tokens give same multiset of norms (order invisible):")
print(f"  Sorted A: {[round(x,3) for x in sorted_a]}")
print(f"  Sorted B: {[round(x,3) for x in sorted_b]}")
print()
print("'I was charged unfairly' and 'Unfairly charged I was' look identical.")
print("FIX: add sinusoidal positional encoding before the first attention layer.")

<!-- DIAGRAM: Heatmap showing sinusoidal positional encoding values. X-axis: embedding dimensions 0 to 31. Y-axis: sequence positions 0 to 15. Color: red for positive values, blue for negative. Alternating sin/cos stripes visible diagonally. Annotate that nearby positions (rows) have similar colors (smooth), while distant positions diverge. This is what the model sees as position information. -->

```mermaid
graph TD
    formula["PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))\nPE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))"]
    formula --> props["Properties of sinusoidal encoding"]
    props --> smooth["Smooth: nearby positions\nhave similar encoding vectors"]
    props --> unique["Unique: every position\nhas a distinct encoding"]
    props --> relative["Relative: PE(pos+k) can be\nexpressed as linear fn of PE(pos)"]
    smooth --> model["Model can learn\n'this token is 3 steps before that one'"]
    unique --> model
    relative --> model
    model --> benefit["No out-of-vocabulary positions:\nworks for any sequence length"]

    style formula fill:#ffd,stroke:#aa3
    style benefit fill:#dfd,stroke:#3a3
```

*Figure: The heatmap shows why sinusoidal encodings work: nearby positions look similar (smooth color transitions row by row), but every position is unique, so the model can distinguish any two positions in the sequence.*

In [ ]:
# Beat 3: Positional encoding from "Attention Is All You Need" - full working demo.
# This is the deterministic sinusoidal encoding, not a learned embedding.
#
# PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
# PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
#
# Why sinusoidal?
#   1. Smooth: nearby positions have similar encodings.
#   2. Generalizes: can encode positions not seen during training.
#   3. Relative: PE(pos+k) is a linear function of PE(pos) - the model can learn offsets.

def positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """
    Compute sinusoidal positional encodings.

    Args:
        max_len: maximum sequence length (number of rows)
        d_model: embedding dimension (number of columns)

    Returns:
        pe: Tensor of shape (max_len, d_model)
    """
    # Position indices: (max_len, 1)
    positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)

    # Dimension indices for half the d_model (sin/cos pairs)
    half_d = d_model // 2
    dim_idx = torch.arange(half_d, dtype=torch.float32).unsqueeze(0)

    # Angle rates: 1 / 10000^(2i / d_model), shape (1, d_model//2)
    angle_rates = 1.0 / (10000.0 ** (dim_idx / d_model))

    # Raw angles: (max_len, d_model//2)
    angles = positions * angle_rates

    # Interleave sin and cos along the d_model axis
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(angles)   # even dims -> sin
    pe[:, 1::2] = torch.cos(angles)   # odd dims -> cos

    return pe

# Compute and visualise for a short sequence
set_seeds(42)
pe = positional_encoding(max_len=50, d_model=64)
print(f"Positional encoding shape: {pe.shape}  (max_len=50, d_model=64)")
print()

# Verify smoothness: cosine similarity between adjacent positions
cos_sims = []
for pos in range(1, 6):
    sim = F.cosine_similarity(pe[pos-1:pos], pe[pos:pos+1]).item()
    cos_sims.append(sim)
print("Cosine similarity between adjacent positions (close to 1.0 means smooth):")
print([f"{s:.4f}" for s in cos_sims])
print(f"Cosine similarity positions 0 and 25 (should be lower):")
print(f"  {F.cosine_similarity(pe[0:1], pe[25:26]).item():.4f}")

# Visualise the encoding pattern
plt.figure(figsize=(10, 4))
plt.imshow(pe[:20, :32].numpy(), aspect="auto", cmap="RdBu_r")
plt.colorbar(label="Encoding value")
plt.xlabel("Embedding dimension")
plt.ylabel("Position")
plt.title("Sinusoidal positional encoding (first 20 positions, first 32 dims)")
plt.tight_layout()
plt.show()
print("Alternating sin/cos stripes are visible across the dimension axis.")

## Lab 1 - Implement PositionalEmbedding (Tier 1 - Guided)

**Time**: 15-20 minutes

### Situation
The Barclays complaints NLP team needs a reusable embedding layer that combines
token embeddings with positional encodings for their new Transformer-based classifier.

### Task
Implement `MyPositionalEmbedding` as an `nn.Module` that:
1. Embeds input token indices using `nn.Embedding`
2. Scales the embedding by sqrt(d_model) - this prevents positional encoding from dominating
3. Adds the precomputed sinusoidal positional encoding

### Action
Complete the two stubs in the class below.

### Result
The verification cell will check output shape and that positional information is included.

### Stretch (fast finishers)
Modify your implementation to accept a `padding_idx` argument in `nn.Embedding` so that the
gradient is masked for padding tokens. The positional encoding for padding positions should
still be added - only the embedding gradient is masked. Test with a batch that includes zeros.

### Homework Extension
In the Transformer paper, the embedding scale `sqrt(d_model)` prevents the positional
encoding from being too small relative to the token embedding. Research:
(a) What happens to training if you remove this scaling?
(b) Some models learn positional encodings instead of using sinusoidal ones. What are the
tradeoffs? Try implementing a `LearnedPositionalEmbedding` module and compare the two on
a small toy sequence classification task.

In [ ]:
# SOLUTION: Lab 1 - PositionalEmbedding with token embedding, scaling, and positional encoding

class MyPositionalEmbedding(nn.Module):
    """
    Combines token embedding with sinusoidal positional encoding.

    Args:
        vocab_size: number of tokens in the vocabulary
        d_model: embedding dimension
        max_len: maximum supported sequence length (default 2048)
    """

    def __init__(self, vocab_size: int, d_model: int, max_len: int = 2048):
        super().__init__()
        self.d_model = d_model

        # SOLUTION Step 1: create an nn.Embedding layer
        # We do NOT use padding_idx here to keep it simple; the exercise does not require it.
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Precompute positional encoding and register as a buffer so it
        # moves to GPU automatically when you call .to(device).
        pe = positional_encoding(max_len, d_model)   # shape: (max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: token indices of shape (batch, seq_len)
        Returns:
            Tensor of shape (batch, seq_len, d_model)
        """
        seq_len = x.size(1)

        # SOLUTION Step 2:
        #   a) embed tokens with self.embedding(x) -> (batch, seq_len, d_model)
        #   b) scale by sqrt(d_model) so the positional encoding does not dominate
        #      (this is the convention from the original Transformer paper)
        #   c) add positional encoding (self.pe[:seq_len] broadcasts over batch)
        embedded = self.embedding(x) * math.sqrt(self.d_model) + self.pe[:seq_len]

        return embedded


# Quick shape check
set_seeds(42)
try:
    my_pe_layer = MyPositionalEmbedding(vocab_size=5000, d_model=128)
    dummy_tokens = torch.randint(1, 5000, (4, 20))  # batch=4, seq_len=20
    out = my_pe_layer(dummy_tokens)
    if out is not None:
        print(f"Output shape: {out.shape}  (expected: torch.Size([4, 20, 128]))")
    else:
        print("Output is None - complete the stubs.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Lab 1 Verification
set_seeds(42)

# Reference implementation to verify against
class _RefPE(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pe", positional_encoding(max_len, d_model))

    def forward(self, x):
        seq_len = x.size(1)
        return self.embedding(x) * math.sqrt(self.d_model) + self.pe[:seq_len]

ref_pe = _RefPE(5000, 128)
test_tokens = torch.randint(1, 5000, (4, 20))

ref_out = ref_pe(test_tokens)

try:
    # Load same weights into student layer
    my_pe_test = MyPositionalEmbedding(5000, 128)
    if my_pe_test.embedding is None:
        print("FAIL: self.embedding is None. Complete Step 1.")
    else:
        my_pe_test.embedding.weight.data = ref_pe.embedding.weight.data.clone()
        my_out = my_pe_test(test_tokens)
        if my_out is None:
            print("FAIL: forward returned None. Complete Step 2.")
        elif my_out.shape != ref_out.shape:
            print(f"FAIL: shape mismatch. Got {my_out.shape}, expected {ref_out.shape}")
        elif torch.allclose(my_out, ref_out, atol=1e-5):
            print("PASS: output matches reference implementation.")
        else:
            diff = (my_out - ref_out).abs().max().item()
            print(f"PARTIAL: shapes match but values differ (max diff={diff:.6f}).")
            print("Check: did you scale by math.sqrt(self.d_model) before adding pe?")
except Exception as e:
    print(f"Error during verification: {e}")

## Beat 3 -- Section 3 - Building the Transformer Block by Block


Now we build the components. We will follow the architecture diagram:
1. Multi-head self-attention with add + layer norm
2. Feed-forward network with add + layer norm
3. Encoder layer (self-attention + FFN)
4. Causal (masked) self-attention for the decoder
5. Cross-attention (encoder-decoder)
6. Decoder layer (masked self-attention + cross-attention + FFN)
7. Full Encoder and Decoder stacks
8. Translator model (Encoder + Decoder + output linear)

We build each component, test its shape, then connect them together.

In [ ]:
# Beat 1: Naive translator WITHOUT positional encoding or multi-head attention.
# We try to build a translator using a single linear layer directly on token embeddings.
# This ignores token order and inter-token relationships entirely.
# Result: the output logits are the same regardless of input order - a broken translator.

set_seeds(42)

NAIVE_SRC_VOCAB = 100
NAIVE_TGT_VOCAB = 120
NAIVE_D = 32

# Naive model: embed -> one linear -> logits. No attention, no positional info.
naive_embed = nn.Embedding(NAIVE_SRC_VOCAB, NAIVE_D)
naive_proj  = nn.Linear(NAIVE_D, NAIVE_TGT_VOCAB)
naive_embed.eval()
naive_proj.eval()

src_order1 = torch.tensor([[1, 2, 3, 4, 5]])   # "mi cuenta fue cobrada incorrectamente"
src_order2 = torch.tensor([[5, 4, 3, 2, 1]])   # same words, reversed

with torch.no_grad():
    logits1 = naive_proj(naive_embed(src_order1))   # (1, 5, 120)
    logits2 = naive_proj(naive_embed(src_order2))   # (1, 5, 120)

print("Beat 1: Naive translator (no attention, no positional encoding)")
print("=" * 64)
print(f"Input order 1: {src_order1.tolist()}")
print(f"Input order 2: {src_order2.tolist()} (reversed)")
print()
print("Predicted token at position 0 for each order:")
pred1 = logits1[0, 0].argmax().item()
pred2 = logits2[0, 0].argmax().item()
print(f"  Order 1, position 0: token {pred1}")
print(f"  Order 2, position 0: token {pred2}")
print()
# Show that the sorted multiset of top predictions is identical
top1 = sorted(logits1[0].argmax(dim=-1).tolist())
top2 = sorted(logits2[0].argmax(dim=-1).tolist())
print(f"Sorted predicted tokens (all positions):")
print(f"  Order 1: {top1}")
print(f"  Order 2: {top2}")
print()
print("Same tokens, different order -> same sorted predictions.")
print("The naive model cannot distinguish 'I charged my account' from 'account my charged I'.")
print()
print("FIX: replace the linear projection with full encoder-decoder Transformer.")
print("     We build this in the cells below.")

## Beat 2 - What a Proper Translator Needs

The naive approach above ignores two things that are essential for translation:

**Positional information**: word order matters. "The bank charged me" and "Me charged the bank"
have the same tokens but different meanings. Without positional encoding the model treats them
identically (as we just saw).

**Token interactions**: translating a word requires understanding its context. "Charge" in
"charge my account" means a financial debit. "Charge" in "I charge you with fraud" is an
accusation. A translation model must read the whole source sequence before generating each
target token.

The Transformer fixes both problems:
- Positional encoding (Section 2) injects order information into the embeddings.
- Multi-head attention (the component we build next) lets every token interact with every
  other token across the full sequence, with learned weights.

We now build the full architecture component by component.

In [ ]:
# Beat 3: Transformer encoder components - full working demo.
# Instructor: walk through each class. Point out shapes at every step.

# ---- PositionalEmbedding (reference for all downstream code) ----

class PositionalEmbedding(nn.Module):
    """Token embedding + sinusoidal positional encoding."""

    def __init__(self, vocab_size: int, d_model: int, max_len: int = 2048):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.register_buffer("pe", positional_encoding(max_len, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        # Scale embedding so positional encoding does not dominate
        emb = self.embedding(x) * math.sqrt(self.d_model)
        return emb + self.pe[:seq_len]


# ---- EncoderLayer ----

class EncoderLayer(nn.Module):
    """
    One encoder layer = Multi-Head Self-Attention + Add+Norm + FFN + Add+Norm.
    Input shape:  (batch, seq_len, d_model)
    Output shape: (batch, seq_len, d_model)  <- same shape in, same shape out
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # batch_first=True means input/output are (batch, seq, d_model) throughout.
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        # Feed-forward: expand by ~4x, ReLU, project back.
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        # Layer norm applied AFTER the residual (post-norm, original paper convention).
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        # Self-attention: Q = K = V = x (each token attends to all other tokens)
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=src_key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))   # residual + norm

        # Feed-forward sublayer
        ff_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ff_out))     # residual + norm

        return x


# ---- Encoder (stack of N EncoderLayers) ----

class Encoder(nn.Module):
    """N stacked EncoderLayers with shared positional embedding."""

    def __init__(
        self, vocab_size: int, d_model: int, num_heads: int,
        d_ff: int, num_layers: int, dropout: float = 0.1
    ):
        super().__init__()
        self.embed = PositionalEmbedding(vocab_size, d_model)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src: torch.Tensor, src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        x = self.embed(src)                          # (batch, seq_len, d_model)
        for layer in self.layers:
            x = layer(x, src_key_padding_mask)
        return self.norm(x)                          # (batch, seq_len, d_model)


# --- Shape verification ---
set_seeds(42)
SRC_VOCAB = 5000
D_MODEL = 128
NUM_HEADS = 4
D_FF = 256
NUM_LAYERS = 2

src_tokens = torch.randint(1, SRC_VOCAB, (3, 20))   # batch=3, seq_len=20
encoder = Encoder(SRC_VOCAB, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS)
encoder_out = encoder(src_tokens)

print("Encoder shape test")
print(f"  Input tokens:   {src_tokens.shape}")
print(f"  Encoder output: {encoder_out.shape}  (expected: [3, 20, 128])")
print(f"  Parameters:     {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
# Beat 3 continued: Decoder components and full Translator model.

class DecoderLayer(nn.Module):
    """
    One decoder layer:
      1. Masked multi-head self-attention (causal - no peeking at future tokens)
      2. Add + Norm
      3. Cross-attention (decoder attends to encoder output)
      4. Add + Norm
      5. FFN
      6. Add + Norm
    Input/output shape: (batch, tgt_len, d_model)
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # Masked self-attention: decoder attends to its own past tokens
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        # Cross-attention: query from decoder, key/value from encoder
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        tgt_key_padding_mask: torch.Tensor = None,
        memory_key_padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        # 1. Masked self-attention: each target token only attends to past tokens
        sa_out, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask,
                                   key_padding_mask=tgt_key_padding_mask)
        tgt = self.norm1(tgt + self.dropout(sa_out))

        # 2. Cross-attention: decoder queries attend to encoder memory (source tokens)
        ca_out, _ = self.cross_attn(tgt, memory, memory,
                                    key_padding_mask=memory_key_padding_mask)
        tgt = self.norm2(tgt + self.dropout(ca_out))

        # 3. Feed-forward
        ff_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout(ff_out))
        return tgt


class Decoder(nn.Module):
    """N stacked DecoderLayers with positional embedding on the target side."""

    def __init__(
        self, vocab_size: int, d_model: int, num_heads: int,
        d_ff: int, num_layers: int, dropout: float = 0.1
    ):
        super().__init__()
        self.embed = PositionalEmbedding(vocab_size, d_model)
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(
        self, tgt: torch.Tensor, memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        tgt_key_padding_mask: torch.Tensor = None,
        memory_key_padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        x = self.embed(tgt)
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask)
        return self.norm(x)


class Translator(nn.Module):
    """
    Full Transformer encoder-decoder model.
    Encoder processes source language tokens.
    Decoder generates target language tokens autoregressively.
    """

    def __init__(
        self, src_vocab: int, tgt_vocab: int, d_model: int,
        num_heads: int, d_ff: int, num_layers: int, dropout: float = 0.1
    ):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout)
        # Final linear: project d_model -> tgt_vocab logits
        self.output_proj = nn.Linear(d_model, tgt_vocab)

    def make_causal_mask(self, sz: int) -> torch.Tensor:
        """Upper triangular bool mask. True positions are masked (set to -inf before softmax)."""
        return torch.triu(torch.ones(sz, sz, dtype=torch.bool), diagonal=1)

    def forward(
        self, src: torch.Tensor, tgt: torch.Tensor,
        src_key_padding_mask: torch.Tensor = None,
        tgt_key_padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        tgt_len = tgt.size(1)
        tgt_mask = self.make_causal_mask(tgt_len).to(src.device)

        memory = self.encoder(src, src_key_padding_mask)
        dec_out = self.decoder(tgt, memory, tgt_mask, tgt_key_padding_mask, src_key_padding_mask)
        return self.output_proj(dec_out)   # (batch, tgt_len, tgt_vocab)


# --- Full model shape test ---
set_seeds(42)
TGT_VOCAB = 6000
model = Translator(SRC_VOCAB, TGT_VOCAB, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS)

src = torch.randint(1, SRC_VOCAB, (3, 20))    # 3 Spanish complaint tickets, 20 tokens
tgt = torch.randint(1, TGT_VOCAB, (3, 18))    # 3 English translations, 18 tokens

logits = model(src, tgt)
print("Full Translator shape test")
print(f"  src:    {src.shape}")
print(f"  tgt:    {tgt.shape}")
print(f"  logits: {logits.shape}  (expected: [3, 18, 6000])")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Discussion (3 minutes)

Look at the parameter count printed above. Discuss with a partner:

1. The model has both encoder and decoder. A modern language model like GPT uses
   ONLY the decoder. What capability does a decoder-only model lose compared to
   an encoder-decoder model? When would you still prefer encoder-decoder?

2. We used `num_heads=4` and `d_model=128`. The original "Attention Is All You Need"
   paper used `num_heads=8` and `d_model=512`. How does increasing heads affect the
   model capacity vs cost tradeoff? Would you use more heads or more layers first?

3. Barclays receives 50,000 complaint tickets per day in Spanish. Would you deploy
   this translator as a real-time endpoint or a batch job? What latency requirements
   drive that decision?

## Lab 2 - Implement DecoderLayer (Tier 2 - Hard)

**Time**: 25-35 minutes

### Situation
The Barclays NLP team has a working Encoder. They need a Decoder that autoregressively
generates English translations from Spanish complaint tickets.

### Task
Implement `build_decoder_layer` so it returns a complete `nn.Module`. Numbered steps
in the starter code guide the structure, but you write each sub-layer yourself.
Use the Beat 3 demo and the architecture diagram as references.

### Stretch
Store the cross-attention weights as `self.last_cross_attn_weights` and plot a heatmap
showing which source tokens the decoder attended to when generating each target token.

### Homework Extension
The original Transformer uses post-norm (Add then LayerNorm). More recent architectures
like LLaMA use pre-norm (LayerNorm then Add). Research the difference and implement a
`PreNormDecoderLayer` variant. Compare training stability on a small toy translation task.

In [ ]:
# SOLUTION: Lab 2 - build_decoder_layer (Tier 2 - Hard)

def build_decoder_layer(d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1) -> nn.Module:
    """
    Build and return a single decoder layer as an nn.Module.

    The layer must implement three sub-layers in order:
        1. Masked multi-head self-attention (causal: future tokens blocked)
        2. Cross-attention (decoder queries attend to encoder output)
        3. Position-wise feed-forward network

    Each sub-layer must be followed by a residual connection and LayerNorm.

    Args:
        d_model:   model embedding dimension
        num_heads: number of attention heads (d_model must be divisible by num_heads)
        d_ff:      hidden dimension of the feed-forward sub-layer
        dropout:   dropout probability applied inside each sub-layer

    Returns:
        An nn.Module whose forward signature is:
            forward(tgt, memory, tgt_mask=None,
                    tgt_key_padding_mask=None,
                    memory_key_padding_mask=None) -> Tensor
        where tgt and the return value have shape (batch, tgt_len, d_model).
    """
    class _DecoderLayer(nn.Module):
        def __init__(self):
            super().__init__()
            # Step 1: masked self-attention (the decoder reads its own past output)
            self.self_attn = nn.MultiheadAttention(
                d_model, num_heads, dropout=dropout, batch_first=True
            )
            # Step 2: cross-attention (decoder queries, encoder keys+values)
            self.cross_attn = nn.MultiheadAttention(
                d_model, num_heads, dropout=dropout, batch_first=True
            )
            # Step 3: feed-forward (expand 4x, ReLU, project back)
            self.ffn = nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(d_ff, d_model),
            )
            # Step 4: norms and dropout for the three residual blocks
            self.norm1   = nn.LayerNorm(d_model)
            self.norm2   = nn.LayerNorm(d_model)
            self.norm3   = nn.LayerNorm(d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(
            self,
            tgt: torch.Tensor,
            memory: torch.Tensor,
            tgt_mask: torch.Tensor = None,
            tgt_key_padding_mask: torch.Tensor = None,
            memory_key_padding_mask: torch.Tensor = None,
        ) -> torch.Tensor:
            # Step 5: masked self-attention -> residual -> dropout -> norm1
            sa_out, _ = self.self_attn(
                tgt, tgt, tgt,
                attn_mask=tgt_mask,
                key_padding_mask=tgt_key_padding_mask,
            )
            tgt = self.norm1(tgt + self.dropout(sa_out))

            # Step 6: cross-attention (Q=tgt, K=V=memory) -> residual -> dropout -> norm2
            ca_out, _ = self.cross_attn(
                tgt, memory, memory,
                key_padding_mask=memory_key_padding_mask,
            )
            tgt = self.norm2(tgt + self.dropout(ca_out))

            # Step 7: feed-forward -> residual -> dropout -> norm3
            ff_out = self.ffn(tgt)
            tgt = self.norm3(tgt + self.dropout(ff_out))

            return tgt

    return _DecoderLayer()


# Quick shape check
set_seeds(42)
try:
    my_dec_layer = build_decoder_layer(d_model=128, num_heads=4, d_ff=256)
    dummy_tgt    = torch.randn(3, 18, 128)
    dummy_memory = torch.randn(3, 20, 128)
    tgt_len = dummy_tgt.size(1)
    causal_mask = torch.triu(torch.ones(tgt_len, tgt_len, dtype=torch.bool), diagonal=1)
    out = my_dec_layer(dummy_tgt, dummy_memory, tgt_mask=causal_mask)
    if out is not None:
        print(f"DecoderLayer output shape: {out.shape}  (expected: torch.Size([3, 18, 128]))")
    else:
        print("Output is None - complete the implementation.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Lab 2 Verification
set_seeds(42)
ref_dec_layer = DecoderLayer(d_model=128, num_heads=4, d_ff=256)

dummy_tgt    = torch.randn(3, 18, 128)
dummy_memory = torch.randn(3, 20, 128)
tgt_len = dummy_tgt.size(1)
causal_mask = torch.triu(torch.ones(tgt_len, tgt_len, dtype=torch.bool), diagonal=1)

ref_out = ref_dec_layer(dummy_tgt, dummy_memory, tgt_mask=causal_mask)

try:
    set_seeds(42)
    my_dec_layer2 = build_decoder_layer(d_model=128, num_heads=4, d_ff=256)
    if my_dec_layer2 is None:
        print("FAIL: build_decoder_layer returned None. Complete the implementation.")
    else:
        my_dec_layer2.load_state_dict(ref_dec_layer.state_dict())
        my_out = my_dec_layer2(dummy_tgt, dummy_memory, tgt_mask=causal_mask)

        if my_out is None:
            print("FAIL: forward returned None. Complete the implementation.")
        elif my_out.shape != ref_out.shape:
            print(f"FAIL: shape mismatch. Got {my_out.shape}, expected {ref_out.shape}")
        elif torch.allclose(my_out, ref_out, atol=1e-5):
            print("PASS: output matches reference implementation numerically.")
        else:
            max_diff = (my_out - ref_out).abs().max().item()
            print(f"PARTIAL: shapes match but values differ (max diff={max_diff:.6f}).")
            print("Check sub-layer order: self-attn -> cross-attn -> FFN?")
            print("Check residual connections and which norm is applied after each.")
except Exception as e:
    print(f"Error during verification: {e}")

## Section 4 - Capstone: Remote GPU Training on SageMaker

You have built the Transformer architecture in this notebook kernel. Now we train
it for real on a remote GPU instance.

This capstone shows the standard SageMaker training pattern: package a training
script, submit a PyTorch estimator job, and retrieve the trained model artifact.
It is the same pattern used to fine-tune models anywhere on SageMaker, so it is
worth seeing once end to end.

What we will do:
1. Write a training script to `scripts_optional_transformers/train.py`
2. Submit a SageMaker PyTorch estimator job (ml.g4dn.xlarge, NVIDIA T4)
3. Wait for the job to complete and retrieve the trained model

The training data is a small synthetic Spanish-English complaint vocabulary
(real translation datasets are too large for a classroom GPU budget).
Job time: approximately 8-12 minutes.

In [ ]:
import os

os.makedirs("scripts_optional_transformers", exist_ok=True)

# Write requirements.txt - only numpy<2 needed; PyTorch 2.8.0 comes from the container.
with open("scripts_optional_transformers/requirements.txt", "w") as f:
    f.write("numpy<2\n")

train_script = r'''
import argparse
import math
import os
import random
import torch
import torch.nn as nn
import numpy as np


def set_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)


def positional_encoding(max_len, d_model):
    positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    half_d = d_model // 2
    dim_idx = torch.arange(half_d, dtype=torch.float32).unsqueeze(0)
    angle_rates = 1.0 / (10000.0 ** (dim_idx / d_model))
    angles = positions * angle_rates
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe


class PositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.register_buffer("pe", positional_encoding(max_len, d_model))

    def forward(self, x):
        seq_len = x.size(1)
        return self.embedding(x) * math.sqrt(self.d_model) + self.pe[:seq_len]


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        a, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + self.dropout(a))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.embed = PositionalEmbedding(vocab_size, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, mask=None):
        x = self.embed(src)
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, tgt_kpm=None, mem_kpm=None):
        a, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_kpm)
        tgt = self.norm1(tgt + self.dropout(a))
        a, _ = self.cross_attn(tgt, memory, memory, key_padding_mask=mem_kpm)
        tgt = self.norm2(tgt + self.dropout(a))
        tgt = self.norm3(tgt + self.dropout(self.ffn(tgt)))
        return tgt


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.embed = PositionalEmbedding(vocab_size, d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask=None, tgt_kpm=None, mem_kpm=None):
        x = self.embed(tgt)
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, tgt_kpm, mem_kpm)
        return self.norm(x)


class Translator(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout)
        self.output_proj = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt):
        tgt_len = tgt.size(1)
        tgt_mask = torch.triu(torch.ones(tgt_len, tgt_len, dtype=torch.bool), diagonal=1).to(src.device)
        memory = self.encoder(src)
        dec_out = self.decoder(tgt, memory, tgt_mask)
        return self.output_proj(dec_out)


def make_batch(batch_size, src_len, tgt_len, src_vocab, tgt_vocab, device):
    src = torch.randint(1, src_vocab, (batch_size, src_len), device=device)
    tgt = torch.randint(1, tgt_vocab, (batch_size, tgt_len), device=device)
    return src, tgt


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    model = Translator(
        src_vocab=args.src_vocab, tgt_vocab=args.tgt_vocab,
        d_model=args.d_model, num_heads=args.num_heads,
        d_ff=args.d_model * 4, num_layers=args.num_layers,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")

    for epoch in range(args.epochs):
        model.train()
        epoch_loss = 0.0
        for step in range(50):
            src, tgt_full = make_batch(args.batch_size, 20, 21, args.src_vocab, args.tgt_vocab, device)
            tgt_in  = tgt_full[:, :-1]
            tgt_out = tgt_full[:, 1:]

            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, args.tgt_vocab), tgt_out.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

            if step % 10 == 0:
                print(f"Epoch {epoch+1}/{args.epochs} Step {step}/50 Loss: {loss.item():.4f}")

        print(f"Epoch {epoch+1} average loss: {epoch_loss / 50:.4f}")

    model_path = os.path.join(os.environ.get("SM_MODEL_DIR", "."), "model.pt")
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--d-model",    type=int,   default=128)
    parser.add_argument("--num-heads",  type=int,   default=4)
    parser.add_argument("--num-layers", type=int,   default=2)
    parser.add_argument("--epochs",     type=int,   default=3)
    parser.add_argument("--batch-size", type=int,   default=16)
    parser.add_argument("--lr",         type=float, default=1e-3)
    parser.add_argument("--src-vocab",  type=int,   default=5000)
    parser.add_argument("--tgt-vocab",  type=int,   default=6000)
    args = parser.parse_args()
    train(args)
'''

with open("scripts_optional_transformers/train.py", "w") as f:
    f.write(train_script)

print("scripts_optional_transformers/train.py        written")
print("scripts_optional_transformers/requirements.txt  written")
print()
print("Contents of scripts_optional_transformers/:")
for fname in sorted(os.listdir("scripts_optional_transformers")):
    size = os.path.getsize(f"scripts_optional_transformers/{fname}")
    print(f"  {fname}  ({size} bytes)")

## Prerequisite for the capstone: AWS / SageMaker credentials

Everything up to this point ran entirely inside this notebook kernel and needed no
cloud access. The capstone is different. It submits a real training job to
Amazon SageMaker, which requires:

- Valid AWS credentials reachable by `boto3` (an attached IAM role on a SageMaker
  notebook instance, or environment credentials elsewhere).
- A SageMaker execution role with permission to create training jobs.
- A writable default S3 bucket for the job output.

If you launched this notebook on a properly provisioned SageMaker notebook
instance, all three are already set up and you can run straight through. If you
are running it somewhere else (a laptop, a plain Jupyter server, an environment
where AWS was never configured), the next cells will fail with a credentials
error.

The cell directly below is a GUARD. It checks for credentials and bucket access
and prints exactly what to do if something is missing, instead of letting you hit
a raw `NoCredentialsError`. Run it first.

If you do not have AWS access, that is fine: the architecture build above is the
core of this deep-dive and is fully complete. You can read the capstone cells
without running them, then come back to the wrap-up.

In [ ]:
# Guard: verify AWS / SageMaker access before submitting the remote training job.
# This cell never raises - it reports status and sets `capstone_ready`.
# Codex R4: a standalone learner with no AWS credentials must get a clear message
# here instead of a raw NoCredentialsError deeper in the capstone.

capstone_ready = True
problems = []

# 0. Is the AWS SDK even installed? A laptop without boto3 would otherwise hit
#    ModuleNotFoundError before the guard logic runs.
try:
    import boto3
    from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError
    _HAS_BOTO3 = True
except ImportError:
    _HAS_BOTO3 = False
    capstone_ready = False
    problems.append("boto3 / botocore not installed (pip install boto3 sagemaker)")

# 1. Are AWS credentials reachable at all?
if _HAS_BOTO3:
    try:
        sts = boto3.client("sts")
        identity = sts.get_caller_identity()
        print(f"AWS credentials OK. Account: {identity['Account']}")
    except (NoCredentialsError, BotoCoreError, ClientError) as e:
        capstone_ready = False
        problems.append(f"No usable AWS credentials: {type(e).__name__}")

# 2. Is there a SageMaker execution role?
try:
    if role is None:
        raise ValueError("role is None")
    print(f"SageMaker execution role OK: {role}")
except Exception as e:
    capstone_ready = False
    problems.append(f"No SageMaker execution role: {e}")

# 3. Is the default bucket reachable and writable?
if _HAS_BOTO3:
    try:
        if bucket is None:
            raise ValueError("bucket is None")
        boto3.client("s3").head_bucket(Bucket=bucket)
        print(f"Default S3 bucket OK: {bucket}")
    except Exception as e:
        capstone_ready = False
        problems.append(f"Default S3 bucket not reachable: {type(e).__name__}")

print()
if capstone_ready:
    print("Capstone prerequisites satisfied. You can run the cells below.")
else:
    print("CAPSTONE CANNOT RUN - missing prerequisites:")
    for p in problems:
        print(f"  - {p}")
    print()
    print("What to do:")
    print("  - On a SageMaker notebook instance: confirm the instance has an")
    print("    attached IAM role with SageMaker and S3 permissions.")
    print("  - Elsewhere: configure AWS credentials (aws configure, or set")
    print("    AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_DEFAULT_REGION).")
    print("  - If you cannot get AWS access, skip the remaining capstone cells.")
    print("    The architecture build above is the core of this deep-dive and is")
    print("    already complete.")

In [ ]:
from sagemaker.pytorch import PyTorch
import time

# Submit the Translator training job to ml.g4dn.xlarge (GPU instance, NVIDIA T4).
# The pattern: source_dir with train.py + requirements.txt -> PyTorch estimator -> .fit()
# This is the standard SageMaker training pattern, reusable for any model.
#
# wait=False: launch the job and return immediately so you can keep reading.
# Re-run the status cell below to check progress.

if not capstone_ready:
    raise RuntimeError(
        "Capstone prerequisites not met - see the guard cell above. "
        "Fix AWS access or skip the remaining capstone cells."
    )

estimator = PyTorch(
    entry_point="train.py",
    source_dir="scripts_optional_transformers",
    role=role,
    framework_version="2.8.0",
    py_version="py312",
    instance_type="ml.g4dn.xlarge",    # GPU instance, NVIDIA T4
    instance_count=1,
    output_path=f"s3://{bucket}/optional-transformers-translator/output",
    base_job_name="barclays-transformer-translator",
    hyperparameters={
        "d-model":    128,
        "num-heads":  4,
        "num-layers": 2,
        "epochs":     3,
        "batch-size": 16,
        "lr":         1e-3,
    },
)

job_name = f"barclays-transformer-{int(time.time())}"

print(f"Submitting training job: {job_name}")
print(f"Instance: ml.g4dn.xlarge (GPU, NVIDIA T4)")
print(f"Estimated time: 8-12 minutes")
print()

estimator.fit(wait=False, job_name=job_name)

training_job_name = estimator.latest_training_job.name
print(f"Job launched: {training_job_name}")
print(f"Monitor at: https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/{training_job_name}")
print()
print("The job is running in the background. Continue with the architecture summary below.")
print("Re-run the cell below to check status, then come back for results.")

In [ ]:
# Safety-net: run this if your kernel restarted after launching the training job.
# SKIP this cell if training_job_name is already defined.
if 'training_job_name' not in dir() or training_job_name is None:
    training_job_name = "<PASTE YOUR JOB NAME HERE>"
    print(f"Using safety-net training_job_name: {training_job_name}")

In [ ]:
# Check training job status. Re-run this cell to refresh.
# describe_training_job raises if the job name is not found yet. boto3 does not
# reliably expose sm_client.exceptions.ResourceNotFound across versions, so we
# catch a broad Exception and treat any failure as "NotFound" (see repo history:
# the same footgun was fixed once already).

sm_client = boto3.client("sagemaker", region_name=region)

def get_job_status(job_name):
    try:
        resp = sm_client.describe_training_job(TrainingJobName=job_name)
        return resp["TrainingJobStatus"], resp.get("SecondaryStatus", "")
    except Exception:
        return "NotFound", ""

status, secondary = get_job_status(training_job_name)
print(f"Job: {training_job_name}")
print(f"Status:           {status}")
print(f"Secondary status: {secondary}")
print()

if status == "InProgress":
    print("Job is running. Estimated completion: 8-12 minutes from launch.")
    print("Continue with the architecture review below while it runs.")
elif status == "Completed":
    resp = sm_client.describe_training_job(TrainingJobName=training_job_name)
    model_uri = resp["ModelArtifacts"]["S3ModelArtifacts"]
    start = resp["TrainingStartTime"]
    end   = resp["TrainingEndTime"]
    dur   = (end - start).total_seconds() / 60
    print(f"Job complete in {dur:.1f} minutes.")
    print(f"Model artifacts: {model_uri}")
elif status == "Failed":
    print("Job failed. Check CloudWatch logs:")
    print(f"  Log group: /aws/sagemaker/TrainingJobs")
    print(f"  Log stream: {training_job_name}/algo-1-*")
else:
    print(f"Status: {status}")

## Wrap-Up - What We Built

### Key takeaways

1. The Transformer removes the recurrent-network bottleneck by replacing sequential
   processing with parallel attention. Every token attends to every other token
   simultaneously.

2. Positional encoding injects order information into an otherwise position-agnostic
   attention mechanism. Sinusoidal encodings generalise to unseen sequence lengths.

3. The encoder-decoder structure naturally separates understanding (encoder) from
   generation (decoder). Cross-attention is the bridge between them.

4. You ran a remote GPU training job on SageMaker ml.g4dn.xlarge. The pattern is
   reusable: write a train.py plus requirements.txt, create a PyTorch estimator,
   call .fit(). The trained model lands in S3 under
   `optional-transformers-translator/output`; download it manually if you want it.

### Homework Extensions

1. Add label smoothing (use `torch.nn.CrossEntropyLoss(label_smoothing=0.1)`) to the
   training script and re-submit the job. Does the loss curve change?

2. Implement greedy decoding: given a trained model and a source sequence, generate
   tokens one at a time until you hit an EOS token or max_len. What challenge arises
   when the source vocabulary does not have real words?

3. Research learning rate warmup as used in the original Transformer paper.
   Implement the warmup schedule and add it to the training script.

### Why this matters when you USE pre-trained models

The architecture you just built is not a toy. The popular pre-trained models are
exactly these blocks:

- A DistilBERT model is a stack of encoder blocks: the encoder half of what you
  built, pre-trained on a large text corpus.
- A T5 or Flan-T5 model is the full encoder-decoder you built, pre-trained at much
  larger scale.
- A GPT-style model is the decoder half, used on its own.

When you load one of these with the transformers library and call it in a few
lines, you now know what is happening inside. That is the payoff of this optional
deep-dive: not a translator you will ship, but the mental model to use, debug, and
choose pre-trained Transformers with confidence.